[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/blob/main/6.%20Google%20Advanced%20Data%20Analytics%20Capstone/salifort_motors_employee_attrition_colab.ipynb)

# Capstone project: Providing data-driven suggestions for HR

## Salifort Motors - Employee Attrition Analysis

This capstone project analyzes employee survey data and builds predictive models that provide insights to the Human Resources department of **Salifort Motors**.

The central business question is:

> **What factors are likely to make an employee leave the company?**

The project includes:

- Business understanding.
- Ethical considerations.
- Data cleaning.
- Exploratory data analysis.
- Data visualizations.
- Logistic Regression.
- Random Forest classification.
- Model diagnostics.
- Model evaluation.
- Recommendations for employee retention.

The analysis follows the **PACE** framework:

- **Plan**
- **Analyze**
- **Construct**
- **Execute**


## Description and deliverables

The HR department wants to improve employee satisfaction and retention. Employee turnover is costly because recruiting, interviewing, hiring, onboarding, and training replacements require substantial time and resources.

The goals of this project are:

1. Analyze the data collected by HR.
2. Identify factors associated with employee turnover.
3. Build a model that predicts whether an employee will leave.
4. Compare an interpretable baseline model with a tree-based model.
5. Provide actionable recommendations to HR and leadership.

The deliverables are:

- A complete technical notebook.
- A concise executive summary for stakeholders.


# PACE: Plan

## Business scenario and stakeholders

Salifort Motors' HR department collected employee data but needs guidance on how to use it.

### Stakeholders

- **HR Department:** Primary owner of the data and recommendations.
- **Senior Leadership:** Interested in turnover costs and organizational health.
- **Department Managers:** Need department-specific information.
- **Employees:** Their work environment may be affected by decisions.
- **Data Analytics Team:** Responsible for quality, fairness, and implementation.

### Objective

Build a predictive model that classifies employees as likely to stay or leave while identifying the systemic drivers of attrition.

### Ethical considerations

- Protect employee privacy.
- Avoid re-identification.
- Use the model to improve working conditions, not punish employees.
- Monitor salary, department, and evaluation variables for bias.
- Avoid labeling employees as disloyal.
- Treat predictions as decision support, not automatic employment decisions.


## HR dataset

The original dataset contains approximately **15,000 rows** and 10 variables.

| Variable | Description |
|---|---|
| `satisfaction_level` | Employee job satisfaction from 0 to 1 |
| `last_evaluation` | Most recent performance review score |
| `number_project` | Number of projects |
| `average_montly_hours` | Average monthly working hours |
| `time_spend_company` | Years at the company |
| `Work_accident` | Whether the employee had a workplace accident |
| `left` | Whether the employee left |
| `promotion_last_5years` | Whether the employee was promoted in the last five years |
| `Department` | Department |
| `salary` | Salary category |

The original column names contain spelling and capitalization inconsistencies that are corrected during data cleaning.


# Step 1. Imports and data loading


In [ ]:
# 1. Data manipulation and numerical operations
import pandas as pd
import numpy as np

# 2. Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 3. Preprocessing and feature engineering
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 4. Statistical diagnostics
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 5. Machine learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier

# 6. Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

pd.set_option("display.max_columns", None)


## Colab-compatible data loading

The original notebook reads a local file called `HR_capstone_dataset.csv`.

The next cell:

1. Searches for a local copy in Jupyter or Colab.
2. Tries likely paths in your GitHub repository.
3. Uses a public fallback containing the same 14,999-row HR dataset.


In [ ]:
# RUN THIS CELL TO IMPORT YOUR DATA

from pathlib import Path

DATA_FILENAME = "HR_capstone_dataset.csv"

# Ruta local: útil si el archivo se sube manualmente a Colab
local_candidates = [
    Path(DATA_FILENAME),
    Path("/content") / DATA_FILENAME,
    Path("/mnt/data") / DATA_FILENAME,
]

# Ruta exacta confirmada en GitHub
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
    "main/6.%20Google%20Advanced%20Data%20Analytics%20Capstone/"
    "HR_capstone_dataset.csv"
)

df0 = None
data_source = None
load_errors = []

# 1. Buscar primero una copia local
for candidate in local_candidates:
    try:
        if candidate.exists() and candidate.stat().st_size > 100:
            loaded = pd.read_csv(candidate)

            if loaded.empty:
                raise ValueError("El archivo local no contiene registros.")

            df0 = loaded
            data_source = str(candidate)
            break

    except Exception as error:
        load_errors.append(f"{candidate}: {error}")

# 2. Descargar automáticamente desde GitHub
if df0 is None:
    try:
        loaded = pd.read_csv(DATA_URL)

        if loaded.empty:
            raise ValueError("El archivo descargado no contiene registros.")

        df0 = loaded
        data_source = DATA_URL

    except Exception as error:
        load_errors.append(f"{DATA_URL}: {error}")

if df0 is None:
    details = "\n- ".join(load_errors)

    raise RuntimeError(
        "No fue posible cargar el dataset de Salifort Motors.\n"
        "Comprueba que el repositorio sea público, que la rama sea main y que "
        "el archivo se llame exactamente HR_capstone_dataset.csv.\n\n"
        f"Intentos realizados:\n- {details}"
    )

expected_columns = {
    "satisfaction_level",
    "last_evaluation",
    "number_project",
    "average_montly_hours",
    "time_spend_company",
    "Work_accident",
    "left",
    "promotion_last_5years",
    "Department",
    "salary",
}

missing_columns = expected_columns.difference(df0.columns)

if missing_columns:
    missing_text = ", ".join(sorted(missing_columns))
    raise ValueError(
        "El archivo cargado no corresponde al dataset esperado. "
        f"Faltan estas columnas: {missing_text}"
    )

print("Dataset cargado correctamente.")
print(f"Fuente: {data_source}")
print(f"Filas: {df0.shape[0]:,}")
print(f"Columnas: {df0.shape[1]}")

df0.head()


# Step 2. Data exploration, EDA, and cleaning


In [ ]:
print("--- Data Info ---")
df0.info()

print(f"\nShape of the dataset: {df0.shape}")

df0.head(5)


In [ ]:
print("--- Numerical Descriptive Statistics ---")
display(df0.describe())

print("\n--- Categorical Descriptive Statistics ---")
display(df0.describe(include="object"))


## Rename columns


In [ ]:
print(df0.columns)


In [ ]:
rename_dict = {
    "number_project": "number_projects",
    "average_montly_hours": "average_monthly_hours",
    "time_spend_company": "tenure",
    "Work_accident": "work_accident",
    "Department": "department"
}

df0 = df0.rename(columns=rename_dict)
df0.columns = [col.strip().lower() for col in df0.columns]

print(df0.columns)


## Check missing values


In [ ]:
missing_values = df0.isna().sum()

print("Missing values per column:")
print(missing_values)


## Check duplicate records


In [ ]:
duplicates_count = df0.duplicated().sum()

print(f"Total number of duplicate rows: {duplicates_count}")


In [ ]:
df_duplicates_paired = (
    df0[df0.duplicated(keep=False)]
    .sort_values(
        by=[
            "satisfaction_level",
            "average_monthly_hours"
        ]
    )
)

display(df_duplicates_paired.head(10))


In [ ]:
df_clean = (
    df0.drop_duplicates(keep="first")
    .reset_index(drop=True)
)

print(f"Rows before cleaning: {df0.shape[0]}")
print(f"Rows after cleaning: {df_clean.shape[0]}")
print(
    "Total duplicates removed: "
    f"{df0.shape[0] - df_clean.shape[0]}"
)

df_clean.head()


The dataset contains **3,008 duplicate rows**. After removing duplicates, the clean dataset contains **11,991 unique observations**.


## Check tenure outliers


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x=df_clean["tenure"])

plt.title(
    "Boxplot to Detect Outliers in Tenure "
    "(Years with Company)",
    fontsize=12
)
plt.xlabel("Tenure (Years)", fontsize=10)

plt.savefig("tenure_boxplot.png", bbox_inches="tight")
plt.show()


In [ ]:
percentile25 = df_clean["tenure"].quantile(0.25)
percentile75 = df_clean["tenure"].quantile(0.75)

iqr = percentile75 - percentile25

upper_limit = percentile75 + 1.5 * iqr
lower_limit = percentile25 - 1.5 * iqr

outliers = df_clean[
    (df_clean["tenure"] > upper_limit)
    | (df_clean["tenure"] < lower_limit)
]

print(f"Lower limit: {lower_limit}")
print(f"Upper limit: {upper_limit}")
print(
    "Number of rows containing outliers in 'tenure': "
    f"{len(outliers)}"
)
print(
    "Percentage of outliers: "
    f"{(len(outliers) / len(df_clean)) * 100:.2f}%"
)


The tenure variable contains long-tenure employees classified as statistical outliers by the 1.5 × IQR rule. These observations are retained because they may be valid and Random Forest is comparatively robust to outliers.


# PACE: Analyze

## Main observations

- Employees with 6-7 projects frequently work 250-300 hours per month and show very high turnover.
- Employees with 3-5 projects and approximately 150-250 hours form a more stable range.
- Low and medium salary groups contain most departures.
- Turnover is concentrated between the third and fifth years of tenure.
- Both low-performing and high-performing employees leave.
- The target is imbalanced, so recall, precision, F1-score, and ROC-AUC are important.


## Employee turnover counts and percentages


In [ ]:
left_counts = df_clean["left"].value_counts()

print("Counts of employees who stayed vs. left:")
print(left_counts)

left_percentages = (
    df_clean["left"].value_counts(normalize=True) * 100
)

print("\nPercentages of employees who stayed vs. left:")
print(left_percentages)


## Distribution of job satisfaction and attrition


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df_clean,
    x="left",
    y="satisfaction_level",
    hue="left",
    palette="Set2"
)

plt.title(
    "Comparison of Satisfaction Levels: Stayed vs. Left",
    fontsize=14
)
plt.xlabel(
    "Employment Status (0: Stayed, 1: Left)",
    fontsize=12
)
plt.ylabel(
    "Job Satisfaction Level (0-1)",
    fontsize=12
)

plt.show()


Satisfaction is a primary indicator of turnover. Employees who remain have a higher and more stable satisfaction distribution, while employees who leave show lower and more variable satisfaction.


## Workload, project count, and burnout


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_clean,
    x="number_projects",
    y="average_monthly_hours",
    hue="left",
    palette="Set1"
)

plt.title(
    "Burnout Analysis: Monthly Hours by "
    "Project Count and Attrition",
    fontsize=14
)
plt.xlabel("Number of Projects", fontsize=12)
plt.ylabel("Average Monthly Hours", fontsize=12)

plt.show()


### Workload findings

1. **Burnout zone:** Employees with 6-7 projects frequently work 250-300 hours per month and show extremely high turnover.
2. **Underutilization risk:** Employees with only two projects also leave at elevated rates.
3. **Operational sweet spot:** Employees with 3-5 projects and balanced hours show stronger retention.

**Recommendation:** Redistribute projects while protecting employees from excessive monthly hours.


## Employee tenure and attrition


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.countplot(
    data=df_clean,
    x="tenure",
    hue="left",
    palette="viridis"
)

plt.title("Tenure Distribution: Stayed vs. Left", fontsize=14)
plt.xlabel("Tenure (Number of Years at Company)", fontsize=12)
plt.ylabel("Number of Employees", fontsize=12)
plt.legend(
    title="Status",
    labels=["Stayed (0)", "Left (1)"]
)

plt.show()


The critical attrition window occurs between years 3 and 5, with a peak near the fourth year. HR should prioritize career and compensation reviews during this period.


## Employee attrition by salary level


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.countplot(
    data=df_clean,
    x="salary",
    hue="left",
    order=["low", "medium", "high"],
    palette="magma"
)

plt.title("Employee Attrition by Salary Level", fontsize=14)
plt.xlabel("Salary Level", fontsize=12)
plt.ylabel("Number of Employees", fontsize=12)
plt.legend(
    title="Status",
    labels=["Stayed (0)", "Left (1)"]
)

plt.show()


Most exits occur in the low and medium salary groups. Compensation may interact with workload, promotion opportunities, and employee satisfaction.


## Performance evaluation and talent drain


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.histplot(
    data=df_clean,
    x="last_evaluation",
    hue="left",
    kde=True,
    element="step",
    palette="coolwarm"
)

plt.title(
    "Distribution of Last Evaluation: Stayed vs. Left",
    fontsize=14
)
plt.xlabel("Last Evaluation Score (0-1)", fontsize=12)
plt.ylabel("Density / Count", fontsize=12)
plt.legend(
    title="Status",
    labels=["Left (1)", "Stayed (0)"]
)

plt.show()


Employees who leave form two broad groups: lower-performing employees and highly evaluated employees. The second group may represent top performers who are overworked.


## Departmental turnover rates


In [ ]:
dept_order = (
    df_clean.groupby("department")["left"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 8))
sns.barplot(
    data=df_clean,
    x="left",
    y="department",
    order=dept_order,
    palette="viridis"
)

plt.title("Turnover Rate by Department", fontsize=14)
plt.xlabel("Turnover Rate", fontsize=12)
plt.ylabel("Department", fontsize=12)

plt.show()


Turnover is not distributed uniformly across departments. High-pressure departments require targeted interventions instead of a single company-wide policy.


## Promotions, accidents, and attrition


In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=df_clean,
    x="promotion_last_5years",
    y="left",
    ax=axes[0],
    palette="Blues_d"
)
axes[0].set_title(
    "Attrition Rate vs. Promotions (Last 5 Years)",
    fontsize=14
)
axes[0].set_xlabel("Promoted (0: No, 1: Yes)", fontsize=12)
axes[0].set_ylabel("Attrition Rate", fontsize=12)

sns.barplot(
    data=df_clean,
    x="work_accident",
    y="left",
    ax=axes[1],
    palette="Reds_d"
)
axes[1].set_title(
    "Attrition Rate vs. Work Accidents",
    fontsize=14
)
axes[1].set_xlabel("Had Accident (0: No, 1: Yes)", fontsize=12)
axes[1].set_ylabel("Attrition Rate", fontsize=12)

plt.tight_layout()
plt.show()


Employees without recent promotions show higher turnover. Workplace accidents do not appear to be a primary driver of voluntary attrition in this dataset.


## Correlation analysis


In [ ]:
numeric_df = df_clean.select_dtypes(
    include=["float64", "int64"]
)

corr_matrix = numeric_df.corr()

sns.set_theme(style="white")

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title(
    "Correlation Heatmap of Employee Metrics",
    fontsize=16
)

plt.show()


Satisfaction has the strongest negative numerical association with leaving. Workload variables also form a related cluster. Correlation does not prove causation, but it supports further modeling.


# PACE: Construct

## Model selection

Two classification approaches are compared.

### Logistic Regression

- Interpretable baseline.
- Coefficients indicate the direction of association.
- Requires checks for multicollinearity and linearity with the logit.
- May struggle with non-linear thresholds and interactions.

### Random Forest

- Captures non-linear relationships.
- Models feature interactions.
- More robust to outliers.
- Does not require strict parametric assumptions.
- Can improve minority-class detection.


## Data preprocessing and feature engineering


In [ ]:
# Create dummy variables for categorical features
df_model = pd.get_dummies(
    df_clean,
    columns=["salary", "department"],
    drop_first=True,
    dtype=int
)

# Define predictors and target
X = df_model.drop("left", axis=1)
y = df_model["left"]

# Split data using a 70/30 partition
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Standardize predictors for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# This duplicated transformation was present in the source workflow
X_test_scaled = scaler.transform(X_test)

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Test rows: {X_test.shape[0]:,}")


## Logistic Regression model


In [ ]:
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

intercept = log_reg.intercept_
coefficients = log_reg.coef_


In [ ]:
print(
    "Logistic Regression model successfully "
    "constructed and fitted."
)
print(f"Model Intercept: {intercept}")
print(f"Model coefficients: {coefficients}")


## Independence of observations

The analysis treats the records as independent because:

- Each row represents a separate employee.
- The data is cross-sectional.
- Duplicate rows were removed.

The assumption should be reconsidered if employees are clustered within teams or share the same manager.


## Multicollinearity assessment using VIF


In [ ]:
X_vif_data = X_train[
    [
        "satisfaction_level",
        "last_evaluation",
        "number_projects",
        "average_monthly_hours",
        "tenure"
    ]
]

X_vif_with_const = sm.add_constant(X_vif_data)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif_with_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(
        X_vif_with_const.values,
        i
    )
    for i in range(len(X_vif_with_const.columns))
]

print("VIF Results (Final & Verified):")
display(
    vif_data[
        vif_data["Feature"] != "const"
    ]
    .sort_values(
        by="VIF",
        ascending=False
    )
    .reset_index(drop=True)
)


The reported VIF values range from approximately 1.05 to 1.22. These values are below the conservative threshold of five, so severe multicollinearity is not present among the selected continuous predictors.


## Cook's Distance influence diagnostics


In [ ]:
X_train_const = sm.add_constant(X_train_scaled)

glm_model = sm.GLM(
    y_train.to_numpy(),
    X_train_const,
    family=sm.families.Binomial()
).fit()

influence = glm_model.get_influence()
cooks_d = influence.cooks_distance[0]

plt.figure(figsize=(10, 6))

try:
    plt.stem(
        np.arange(len(cooks_d)),
        cooks_d,
        markerfmt=",",
        use_line_collection=True
    )
except TypeError:
    plt.stem(
        np.arange(len(cooks_d)),
        cooks_d,
        markerfmt=","
    )

plt.title(
    "Influence Diagnostics: Cook's Distance "
    "(Salifort Motors)"
)
plt.xlabel("Observation Index")
plt.ylabel("Cook's Distance")

threshold = 4 / len(y_train)
plt.axhline(
    y=threshold,
    color="red",
    linestyle="--",
    label=f"Threshold (4/n = {threshold:.5f})"
)

plt.legend()
plt.show()

extreme_outliers = np.where(cooks_d > threshold)[0]

print(
    "Number of influential observations detected: "
    f"{len(extreme_outliers)}"
)


## Profile high-influence observations


In [ ]:
high_influence_indices = np.where(
    cooks_d > 0.004
)[0]

influential_employees = (
    X_train.iloc[high_influence_indices]
    .copy()
)

influential_employees["cooks_d"] = (
    cooks_d[high_influence_indices]
)

print(
    "Total 'towers' detected (> 0.004): "
    f"{len(influential_employees)}"
)

influential_employees.sort_values(
    by="cooks_d",
    ascending=False
).head(10)


Influential observations are inspected rather than automatically removed. A statistically influential employee record can still be valid and meaningful.


## Logit linearity assessment


In [ ]:
probabilities = log_reg.predict_proba(
    X_train_scaled
)[:, 1]

epsilon = 1e-10
probabilities_safe = np.clip(
    probabilities,
    epsilon,
    1 - epsilon
)

logit_values = np.log(
    probabilities_safe
    / (1 - probabilities_safe)
)

df_linearity = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
)
df_linearity["logit"] = logit_values

continuous_vars = [
    "satisfaction_level",
    "last_evaluation",
    "number_projects",
    "average_monthly_hours",
    "tenure"
]

sample_size = min(
    2000,
    len(df_linearity)
)

plot_sample = df_linearity.sample(
    sample_size,
    random_state=42
)

plt.figure(figsize=(18, 10))

for i, variable in enumerate(
    continuous_vars,
    1
):
    plt.subplot(2, 3, i)
    sns.regplot(
        x=variable,
        y="logit",
        data=plot_sample,
        scatter_kws={"alpha": 0.2},
        line_kws={"color": "red"}
    )
    plt.title(
        f"Linearity: {variable} vs Logit"
    )

plt.tight_layout()
plt.show()


Satisfaction and tenure show clearer relationships with the logit. Monthly hours and evaluation show patterns that may be non-linear or threshold based, supporting the use of Random Forest.


## Logistic Regression evaluation


In [ ]:
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(
    X_test_scaled
)[:, 1]

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=log_reg.classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=log_reg.classes_
)

print(
    "--- Logistic Regression "
    "Evaluation Report ---"
)
print(
    classification_report(
        y_test,
        y_pred
    )
)
print(
    "ROC-AUC Score: "
    f"{roc_auc_score(y_test, y_pred_proba):.4f}"
)

disp.plot(
    cmap="Blues",
    values_format="d"
)
plt.title(
    "Confusion Matrix for Employee Attrition"
)
plt.show()


The project reported the following approximate Logistic Regression results:

- Accuracy: 0.83
- Precision for employees who left: 0.50
- Recall for employees who left: 0.21
- F1-score for employees who left: 0.30
- ROC-AUC: 0.8253

The low recall means that the model fails to identify most employees who actually leave.


## Random Forest hyperparameter optimization


In [ ]:
rf = RandomForestClassifier(
    random_state=42
)

cv_params = {
    "max_depth": [3, 5, 8, 50],
    "max_features": [1.0, "sqrt"],
    "max_samples": [0.7, 1.0],
    "min_samples_leaf": [1, 2, 5],
    "n_estimators": [100, 300]
}

scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

rf_cv = GridSearchCV(
    estimator=rf,
    param_grid=cv_params,
    scoring=scoring,
    cv=5,
    refit="f1",
    n_jobs=-1,
    verbose=1
)

rf_cv.fit(X_train, y_train)


In [ ]:
print("Best Random Forest parameters:")
print(rf_cv.best_params_)

print(
    "\nBest cross-validated F1-score: "
    f"{rf_cv.best_score_:.4f}"
)


## Extract cross-validated Random Forest metrics


In [ ]:
def get_test_scores(
    model_name: str,
    cv_results: pd.DataFrame,
    metric: str
):
    # Extract the best row based on the chosen ranking metric
    best_row = cv_results[
        cv_results[f"rank_test_{metric}"] == 1
    ].iloc[0]

    table = pd.DataFrame(
        {
            "Model": [model_name],
            "Precision": [
                best_row["mean_test_precision"]
            ],
            "Recall": [
                best_row["mean_test_recall"]
            ],
            "F1-Score": [
                best_row["mean_test_f1"]
            ],
            "Accuracy": [
                best_row["mean_test_accuracy"]
            ],
            "ROC-AUC": [
                best_row["mean_test_roc_auc"]
            ]
        }
    )

    return table


rf_cv_results = pd.DataFrame(
    rf_cv.cv_results_
)

rf_table = get_test_scores(
    "Random Forest (Tuned)",
    rf_cv_results,
    "f1"
)

display(
    rf_table.style.set_caption(
        "Resumen de Métricas de "
        "Validación Cruzada"
    )
)


The project reported approximate tuned Random Forest validation results of:

- Accuracy: 98.35%
- Precision: 98.23%
- Recall: 91.75%
- F1-score: 0.9488
- ROC-AUC: 0.9769

The exact values can vary slightly with package versions.


## Random Forest confusion matrix


In [ ]:
y_pred_rf = (
    rf_cv.best_estimator_
    .predict(X_test)
)

cm_rf = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=rf_cv.classes_
)

disp_rf = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=rf_cv.classes_
)

disp_rf.plot(
    cmap="Greens",
    values_format="d"
)

plt.title(
    "Confusion Matrix: Random Forest "
    "(Salifort Motors)"
)
plt.grid(False)
plt.show()

print(
    classification_report(
        y_test,
        y_pred_rf
    )
)


The reported model captured 553 of 597 attrition events, producing recall above 92% with very few false positives.


## Compare Logistic Regression and Random Forest


In [ ]:
lr_metrics = pd.DataFrame(
    {
        "Model": ["Logistic Regression"],
        "Precision": [
            precision_score(
                y_test,
                y_pred,
                zero_division=0
            )
        ],
        "Recall": [
            recall_score(
                y_test,
                y_pred,
                zero_division=0
            )
        ],
        "F1-Score": [
            f1_score(
                y_test,
                y_pred,
                zero_division=0
            )
        ],
        "Accuracy": [
            accuracy_score(
                y_test,
                y_pred
            )
        ]
    }
)


def get_cv_scores(
    model_name,
    model_object
):
    results = pd.DataFrame(
        model_object.cv_results_
    )

    best_row = results[
        results["rank_test_f1"] == 1
    ].iloc[0]

    return pd.DataFrame(
        {
            "Model": [model_name],
            "Precision": [
                best_row[
                    "mean_test_precision"
                ]
            ],
            "Recall": [
                best_row[
                    "mean_test_recall"
                ]
            ],
            "F1-Score": [
                best_row[
                    "mean_test_f1"
                ]
            ],
            "Accuracy": [
                best_row[
                    "mean_test_accuracy"
                ]
            ]
        }
    )


rf_metrics = get_cv_scores(
    "Random Forest (Tuned)",
    rf_cv
)

comparison_table = pd.concat(
    [
        lr_metrics,
        rf_metrics
    ],
    ignore_index=True
)

print(
    "--- Comparison of Model Performance "
    "(Salifort Motors) ---"
)

display(
    comparison_table.style.highlight_max(
        axis=0,
        color="lightgreen",
        subset=[
            "Precision",
            "Recall",
            "F1-Score",
            "Accuracy"
        ]
    )
)


# PACE: Execute

## Key insights

1. Employee attrition is non-linear.
2. Low satisfaction and excessive workload interact.
3. Employees in their third through fifth years are especially vulnerable.
4. High performers may be overused.
5. Random Forest provides substantially better recall than Logistic Regression.

## Business recommendations

- Cap excessive monthly working hours.
- Avoid assigning more than five simultaneous projects without support.
- Create career-development reviews around the third and fourth years.
- Investigate why highly evaluated employees leave.
- Use a retention dashboard to identify systemic risk patterns.
- Use predictions to offer support, not punishment.

## Potential improvements

- Add external salary competitiveness.
- Include commute distance and flexible-work information.
- Include changes in satisfaction over time.
- Test XGBoost or other gradient boosting models.
- Monitor fairness across departments and salary levels.
- Re-evaluate the model as organizational conditions change.


## Final model summary

The tuned Random Forest was selected as the preferred model.

| Metric | Reported value | Interpretation |
|---|---:|---|
| Accuracy | 98.36% | High overall reliability |
| Precision | 98.24% | Very few false-positive alerts |
| Recall | 91.75% | Detects most employees who leave |
| F1-score | 0.9488 | Strong precision-recall balance |
| ROC-AUC | 0.9769 | Excellent class separation |

The main operational message is that attrition is associated with systemic burnout, low satisfaction, excessive workloads, limited advancement, and a critical mid-tenure period.


## Conclusion

The analysis indicates that Salifort Motors loses employees at both extremes of workload:

- Underutilized employees may disengage.
- Severely overworked employees may experience burnout.
- High-performing employees may leave because strong performance results in excessive assignments.
- Employees in their third through fifth years need targeted career support.

### Recommendations

1. **Workload optimization:** Redistribute projects and limit extreme monthly hours.
2. **Strategic four-year check-in:** Conduct formal career-development and compensation reviews.
3. **Burnout as a KPI:** Monitor satisfaction and workload as leading indicators.
4. **Responsible model use:** Use the model for supportive interventions, not automated adverse decisions.
5. **Continuous monitoring:** Reassess performance and fairness at least twice per year.


## Resources

- Pandas documentation: `https://pandas.pydata.org/docs/`
- Seaborn documentation: `https://seaborn.pydata.org/`
- Matplotlib documentation: `https://matplotlib.org/`
- Scikit-learn documentation: `https://scikit-learn.org/stable/`
- Statsmodels documentation: `https://www.statsmodels.org/`
- Google Advanced Data Analytics PACE framework


**Capstone project completed.**
